In [7]:
import pandas as pd
import sys
import subprocess
import os
df = pd.read_csv("cleaned_data.csv")
df['is_night'] = df['transaction_time'].apply(lambda x: 1 if x <= 6 or x >= 22 else 0)
df['amount_per_age'] = df['transaction_amount'] / (df['customer_age'] + 1)
df['is_high_risk_merchant'] = df['merchant_category'].apply(lambda x: 1 if x in [1, 3] else 0)
X = df.drop(["is_fraud", "transaction_id"], axis=1, errors='ignore')
y = df["is_fraud"]

In [8]:
from sklearn.model_selection import train_test_split

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]


X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)



In [9]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier

models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
}


In [10]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

for name, model in models.items():
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test)
    print(f" {name} Results")
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, model.predict_proba(X_test)[:, 1]))
    print("—" * 40)


 RandomForest Results
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.97      0.98     14550
           1       0.30      0.45      0.36       450

    accuracy                           0.95     15000
   macro avg       0.64      0.71      0.67     15000
weighted avg       0.96      0.95      0.96     15000

Confusion Matrix:
 [[14087   463]
 [  249   201]]
ROC AUC Score: 0.6996287896143566
————————————————————————————————————————


c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


 LogisticRegression Results
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.74      0.84     14550
           1       0.06      0.49      0.10       450

    accuracy                           0.73     15000
   macro avg       0.52      0.62      0.47     15000
weighted avg       0.95      0.73      0.82     15000

Confusion Matrix:
 [[10797  3753]
 [  228   222]]
ROC AUC Score: 0.6506002290950745
————————————————————————————————————————
 GradientBoosting Results
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.89      0.94     14550
           1       0.12      0.48      0.20       450

    accuracy                           0.88     15000
   macro avg       0.55      0.69      0.57     15000
weighted avg       0.96      0.88      0.91     15000

Confusion Matrix:
 [[12996  1554]
 [  232   218]]
ROC AUC Score: 0.7094370370370371
———————————————————————————————————

c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:45:52] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [11]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
import pandas as pd
import joblib

df = pd.read_csv("cleaned_data.csv")  
X = df.drop(["is_fraud", "transaction_id"], axis=1, errors='ignore')
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

models = {
    "RandomForest": RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    "LogisticRegression": LogisticRegression(max_iter=1000, random_state=42),
    "GradientBoosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42, n_jobs=-1)
}

for name, model in models.items():
    model.fit(X_resampled, y_resampled)
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    print(f" {name} ")
    print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
    print("Classification Report:\n", classification_report(y_test, y_pred))
    print("ROC AUC Score:", roc_auc_score(y_test, y_proba))
    print("-" * 50)

    joblib.dump(model, f"{name}_model.pkl")


 RandomForest 
Confusion Matrix:
 [[13260  1290]
 [  234   216]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.91      0.95     14550
           1       0.14      0.48      0.22       450

    accuracy                           0.90     15000
   macro avg       0.56      0.70      0.58     15000
weighted avg       0.96      0.90      0.92     15000

ROC AUC Score: 0.7086996563573884
--------------------------------------------------
 LogisticRegression 
Confusion Matrix:
 [[10295  4255]
 [  186   264]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.71      0.82     14550
           1       0.06      0.59      0.11       450

    accuracy                           0.70     15000
   macro avg       0.52      0.65      0.46     15000
weighted avg       0.95      0.70      0.80     15000

ROC AUC Score: 0.6821018709431081
-------------------------------------------

c:\Users\MSI\AppData\Local\Programs\Python\Python313\Lib\site-packages\xgboost\training.py:183: UserWarning: [13:46:09] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


In [12]:
import joblib
from sklearn.ensemble import VotingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.model_selection import train_test_split
import pandas as pd

df = pd.read_csv("cleaned_data.csv")
X = df.drop(["is_fraud", "transaction_id"], axis=1, errors='ignore')
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)
from imblearn.over_sampling import SMOTE

sm = SMOTE(random_state=42)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)

rf = joblib.load("RandomForest_model.pkl")
lr = joblib.load("LogisticRegression_model.pkl")
gb = joblib.load("GradientBoosting_model.pkl")
ada = joblib.load("AdaBoost_model.pkl")
xgb = joblib.load("XGBoost_model.pkl")

ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('lr', lr),
        ('gb', gb),
        ('ada', ada),
        ('xgb', xgb)
    ],
    voting='soft',
    n_jobs=-1
)

ensemble_model.fit(X_resampled, y_resampled)

y_pred = ensemble_model.predict(X_test)
y_proba = ensemble_model.predict_proba(X_test)[:, 1]

print(" Ensemble Voting Classifier Results:")
print("Confusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC AUC Score:", roc_auc_score(y_test, y_proba))

joblib.dump(ensemble_model, "ensemble_model.pkl")


 Ensemble Voting Classifier Results:
Confusion Matrix:
 [[12768  1782]
 [  224   226]]
Classification Report:
               precision    recall  f1-score   support

           0       0.98      0.88      0.93     14550
           1       0.11      0.50      0.18       450

    accuracy                           0.87     15000
   macro avg       0.55      0.69      0.56     15000
weighted avg       0.96      0.87      0.90     15000

ROC AUC Score: 0.7163943489881633


['ensemble_model.pkl']

In [13]:
ensemble_model.predict(X_test)
classification_report(y_test, y_pred)


'              precision    recall  f1-score   support\n\n           0       0.98      0.88      0.93     14550\n           1       0.11      0.50      0.18       450\n\n    accuracy                           0.87     15000\n   macro avg       0.55      0.69      0.56     15000\nweighted avg       0.96      0.87      0.90     15000\n'

In [14]:

from sklearn.metrics import classification_report, roc_auc_score
import numpy as np

y_scores = ensemble_model.predict_proba(X_test)[:, 1]

thresholds = np.arange(0.1, 0.9, 0.05)
for thresh in thresholds:
    y_pred_custom = (y_scores > thresh).astype(int)
    report = classification_report(y_test, y_pred_custom, output_dict=True)
    roc_auc = roc_auc_score(y_test, y_scores)
    print(f" Threshold: {thresh:.2f}")
    print(f"Precision: {report['1']['precision']:.2f} | Recall: {report['1']['recall']:.2f} | F1: {report['1']['f1-score']:.2f} | ROC AUC: {roc_auc:.4f}")
    print("-" * 50)


 Threshold: 0.10
Precision: 0.03 | Recall: 0.97 | F1: 0.06 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.15
Precision: 0.03 | Recall: 0.87 | F1: 0.07 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.20
Precision: 0.04 | Recall: 0.78 | F1: 0.07 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.25
Precision: 0.04 | Recall: 0.69 | F1: 0.08 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.30
Precision: 0.05 | Recall: 0.63 | F1: 0.10 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.35
Precision: 0.06 | Recall: 0.59 | F1: 0.11 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.40
Precision: 0.08 | Recall: 0.55 | F1: 0.13 | ROC AUC: 0.7164
--------------------------------------------------
 Threshold: 0.45
Precision: 0.09 | Recall: 0.52 | F1: 0.15 | ROC AUC: 0.7164
---------------------------

In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score
from sklearn.ensemble import VotingClassifier
from imblearn.over_sampling import SMOTE
import joblib

df = pd.read_csv("cleaned_data.csv")

df['is_night'] = df['transaction_time'].apply(lambda x: 1 if x <= 6 or x >= 22 else 0)
df['amount_per_age'] = df['transaction_amount'] / (df['customer_age'] + 1)
df['is_high_risk_merchant'] = df['merchant_category'].apply(lambda x: 1 if x in [1, 3] else 0)

X = df.drop(['is_fraud', 'transaction_id'], axis=1, errors='ignore')
y = df['is_fraud']

X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, test_size=0.3, random_state=42)

sm = SMOTE(random_state=42)
X_train_smote, y_train_smote = sm.fit_resample(X_train, y_train)

rf = joblib.load("RandomForest_model.pkl")
lr = joblib.load("LogisticRegression_model.pkl")
gb = joblib.load("GradientBoosting_model.pkl")
ada = joblib.load("AdaBoost_model.pkl")
xgb = joblib.load("XGBoost_model.pkl")

ensemble_model = VotingClassifier(
    estimators=[
        ('rf', rf),
        ('lr', lr),
        ('gb', gb),
        ('ada', ada),
        ('xgb', xgb)
    ],
    voting='soft',
    n_jobs=-1
)

ensemble_model.fit(X_train_smote, y_train_smote)

y_pred = ensemble_model.predict(X_test)
y_proba = ensemble_model.predict_proba(X_test)[:, 1]

print(" CONFUSION MATRIX:\n", confusion_matrix(y_test, y_pred))
print("\n CLASSIFICATION REPORT:\n", classification_report(y_test, y_pred))
print(" ROC AUC SCORE:", roc_auc_score(y_test, y_proba))
joblib.dump(ensemble_model, "ensemble_model.pkl")


 CONFUSION MATRIX:
 [[13696   854]
 [  243   207]]

 CLASSIFICATION REPORT:
               precision    recall  f1-score   support

           0       0.98      0.94      0.96     14550
           1       0.20      0.46      0.27       450

    accuracy                           0.93     15000
   macro avg       0.59      0.70      0.62     15000
weighted avg       0.96      0.93      0.94     15000

 ROC AUC SCORE: 0.7057518136693395


['ensemble_model.pkl']